# 05 — Resultados Completos, `AffinePNN` Pura, `AffinePNN` × `DNN_AN` e Exportação `ONNX`

Este notebook reúne **todos os resultados e figuras** da análise, divididos em duas partes claras (**desempenho intrínseco da `AffinePNN` pura** e **comparação controlada contra a `DNN_AN` da AN-25-103**), além da exportação completa para produção/CMSSW:

1. **Métricas globais multiclasse e binárias por $\kappa_\lambda \in \{0, 1, 2.45, 5\}$** (`AUC_w`, `PR-AUC`, `Precision`, `Recall`, `F1-score`, `Melhor F1`, `Significance Ratio` e incerteza via *bootstrap*).
2. **Análise Completa da Rede `AffinePNN` Pura (Standalone):**
   - **Curvas ROC** para cada $\kappa_\lambda$ (linear c/ zoom + rejeição $\epsilon_B$ em log) e todas as hipóteses sobrepostas.
   - **Scores da rede $P(\text{HH})$ vs Eventos em escala log** para cada $\kappa_\lambda$ (eventos MC brutos, rendimentos físicos $\sum w_{\text{phys}}$ e fundos empilhados).
   - **Curvas Precision-Recall** (com contornos iso-F1) e resumo de **Precision, Recall e F1-score**.
   - **Precision, Recall e F1-score vs Score da rede $P(\text{HH})$** — **plots separados** (um para cada métrica) e **um plot único comparando todos juntos**.
   - **Matrizes de confusão multiclasse** (`DY`, `ttbar`, `HH`) por $\kappa_\lambda$.
3. **Comparação Direta `AffinePNN` vs `DNN_AN` (`Single-NN`):**
   - **Curvas ROC comparativas** para cada $\kappa_\lambda$ e painel $2 \times 4$.
   - **Scores da rede vs Eventos em escala log** para cada $\kappa_\lambda$ (`AffinePNN` vs `DNN_AN`).
   - **Curvas Precision-Recall (iso-F1)** e **barras comparativas de Precision, Recall e F1-score**.
   - **Precision, Recall e F1-score vs Score da rede** — **plots separados** e **plot único comparando todos juntos** (`AffinePNN` vs `DNN_AN`).
   - **Matrizes de confusão comparativas**, **Significance Ratio $\sigma_{\text{ratio}}(t)$** (Anzalone Eq. 4) e **rendimentos físicos** ($S/B$, $S/\sqrt{B}$).
4. **Varredura contínua fina em $\kappa_\lambda \in [-2, 8]$** (`Continuous Kappa Scan`).
5. **Exportação completa e autossuficiente (`ONNX`, `ONNX End-to-End`, `SavedModel`, `.keras`, `.weights.h5`, `.npz`)** e inferência direta com `ONNXModelPredictor`.

> **Trava de cegamento do conjunto de teste:** mantenha `EVAL_SPLIT = "val"` enquanto iterar sobre arquitetura ou hiperparâmetros. Mude para `EVAL_SPLIT = "test"` e `UNLOCK_TEST_SET = True` **uma única vez**, com tudo congelado, para gerar os números finais da dissertação/artigo.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data.preprocessing import prepare_datasets_from_config
from src.evaluation.evaluate import run_full_evaluation
from src.evaluation.kappa_scan import continuous_kappa_scan, kappa_response_matrix
from src.evaluation.performance_plots import (
    plot_confusion_matrices,
    plot_metric_vs_score_kappas,
    plot_score_events_log,
    plot_score_events_log_kappas,
    plot_working_point_summary,
    working_point_table,
)
from src.evaluation.plotting import (
    plot_continuous_kappa_scan,
    plot_kappa_response_heatmap,
    plot_metrics_bars_cmp,
    plot_physical_yields_and_significance,
    plot_pr_curves_and_f1_summary,
    plot_prf1_vs_score_all_in_one,
    plot_prf1_vs_score_separate,
    plot_roc_and_pr_grid,
    plot_roc_per_kappa_individual,
    plot_scores_vs_events_log_per_kappa,
    plot_signal_eff_vs_bkg_rejection,
    plot_significance_ratio,
)
from src.models.export import ONNXModelPredictor, export_complete_model_bundle
from src.training.train import load_model_bundle
from src.utils.config import load_config
from src.utils.paths import OutputPaths
from src.utils.seed import configure_gpu_memory_growth, set_all_seeds

configure_gpu_memory_growth()
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 220)

In [ ]:
CUSTOM_OUTPUT_DIR = None   # Ex.: "/eos/user/m/mgeralde/Analysis_hh2bbtautau/PNN/affinepnn_outputs"
MODEL_NAMES = ["AffinePNN", "DNN_AN"]

EVAL_SPLIT = "val"         # "val" | "test"
UNLOCK_TEST_SET = False    # Coloque True APENAS na avaliação final congelada!

if EVAL_SPLIT == "test" and not UNLOCK_TEST_SET:
    raise RuntimeError("Conjunto 'test' bloqueado! Defina UNLOCK_TEST_SET = True apenas com a configuração congelada.")

cfg = load_config(PROJECT_ROOT / "configs" / "default.yaml")
if CUSTOM_OUTPUT_DIR:
    cfg["paths"]["output_dir"] = CUSTOM_OUTPUT_DIR

set_all_seeds(cfg["seed"])
out_paths = OutputPaths.from_root(cfg["paths"]["output_dir"], create=True)
ckpt_dir = out_paths.checkpoints
plots_dir = out_paths.plots / EVAL_SPLIT
pure_dir = plots_dir / "affine_pure"
cmp_dir = plots_dir / "comparison"
for d in (plots_dir, pure_dir, cmp_dir):
    d.mkdir(parents=True, exist_ok=True)

models = {}
histories = {}
metas = {}
for name in MODEL_NAMES:
    m, h, meta = load_model_bundle(ckpt_dir, name)
    models[name] = m
    if h is not None:
        histories[name] = h
    if meta is not None:
        metas[name] = meta
    print(f"Carregado {name}: {m.count_params():,d} parâmetros")

data = prepare_datasets_from_config(cfg, existing_checkpoints_dir=ckpt_dir, verbose=True)
kgrid = cfg["physics"]["kappa_grid"]
cls_labels = cfg["physics"]["class_labels"]

## 1. Avaliação Completa (Multiclasse + Binária por $\kappa_\lambda$ + Pontos de Trabalho + Bootstrap)

In [ ]:
eval_res = run_full_evaluation(
    models=models,
    data=data,
    cfg=cfg,
    split=EVAL_SPLIT,
    metrics_dir=out_paths.metrics,
    verbose=True,
)

if eval_res["comparison"] is not None:
    print("\nTabela Comparativa Direta (AffinePNN vs DNN_AN):")
    display(eval_res["comparison"].round(4))

wp_df = working_point_table(eval_res["scores"], models=MODEL_NAMES, kappas=kgrid)
wp_df.to_csv(out_paths.metrics / f"working_points_{EVAL_SPLIT}.csv", index=False)
print("\nPontos Ótimos de F1-score e Rejeição de Fundo 1/ε_B:")
display(wp_df.round(4))

---
# Parte A — Desempenho da Rede `AffinePNN` Pura (Standalone)

## 2.1 `AffinePNN` Pura — Curvas ROC para cada $\kappa_\lambda$ e Sobreposição Global

In [ ]:
affine_only_scores = {"AffinePNN": eval_res["scores"]["AffinePNN"]}

_ = plot_roc_per_kappa_individual(
    scores=affine_only_scores,
    kappas=kgrid,
    title_prefix="AffinePNN Pura",
    save_dir=pure_dir / "roc_per_kappa",
    show=True,
)

## 2.2 `AffinePNN` Pura — Scores da Rede $P(\text{HH})$ vs Eventos em Escala Log por $\kappa_\lambda$

Exibe para cada hipótese $\kappa_\lambda$ tanto a contagem de eventos MC brutos quanto o rendimento físico ponderado ($\sum w_{\text{phys}}^+$) em escala logarítmica, além do painel $2 \times 2$ com fundos empilhados (`DY + ttbar`) vs sinal `HH`.

In [ ]:
_ = plot_scores_vs_events_log_per_kappa(
    scores=affine_only_scores,
    class_labels=cls_labels,
    kappas=kgrid,
    save_dir=pure_dir / "scores_vs_events_log",
    show=True,
)

_ = plot_score_events_log_kappas(
    scores=eval_res["scores"],
    model_name="AffinePNN",
    class_labels=cls_labels,
    kappas=kgrid,
    weighted=True,
    save_path=pure_dir / "scores_stacked_yields_log_AffinePNN.png",
    show=True,
)

## 2.3 `AffinePNN` Pura — Curvas Precision-Recall (com Contornos Iso-F1) e Resumo Precision / Recall / F1

In [ ]:
_ = plot_pr_curves_and_f1_summary(
    scores=affine_only_scores,
    kappas=kgrid,
    save_dir=pure_dir,
    show=True,
)

## 2.4 `AffinePNN` Pura — Precision, Recall e F1-score vs Score da Rede $P(\text{HH})$
- **Plots separados:** um painel dedicado exclusivamente a `Precision vs Score`, outro a `Recall vs Score` e outro a `F1-score vs Score` (além da sobreposição entre todos os $\kappa_\lambda$).
- **Plot único comparando todos juntos:** `Precision + Recall + F1-score` no mesmo gráfico para cada $\kappa_\lambda$.

In [ ]:
# 1) Plots SEPARADOS (Precision vs Score, Recall vs Score, F1-score vs Score)
_ = plot_prf1_vs_score_separate(
    scores=affine_only_scores,
    kappas=kgrid,
    save_dir=pure_dir / "prf1_vs_score_separate",
    show=True,
)

# Sobreposição direta de todas as hipóteses κ_λ para cada métrica isolada
for metric_name in ("precision", "recall", "f1"):
    _ = plot_metric_vs_score_kappas(
        scores=eval_res["scores"],
        model_name="AffinePNN",
        metric=metric_name,
        kappas=kgrid,
        save_path=pure_dir / "prf1_vs_score_separate" / f"{metric_name}_vs_score_all_kappas_AffinePNN.png",
        show=True,
    )

# 2) Plot ÚNICO comparando TODOS juntos (Precision + Recall + F1 vs Score)
_ = plot_prf1_vs_score_all_in_one(
    scores=affine_only_scores,
    kappas=kgrid,
    save_path=pure_dir / "prf1_vs_score_all_in_one_AffinePNN.png",
    show=True,
)

## 2.5 `AffinePNN` Pura — Matrizes de Confusão Multiclasse (`DY`, `ttbar`, `HH`) por $\kappa_\lambda$

In [ ]:
for k in kgrid:
    tag = str(k).replace(".", "p")
    _ = plot_confusion_matrices(
        scores=affine_only_scores,
        kappa=float(k),
        class_labels=cls_labels,
        models=["AffinePNN"],
        save_path=pure_dir / f"confusion_matrix_AffinePNN_kl_{tag}.png",
        show=True,
    )

---
# Parte B — Comparação Direta `AffinePNN` vs `DNN_AN` (`Single-NN` da AN-25-103)

## 3.1 `AffinePNN` vs `DNN_AN` — Curvas ROC para cada $\kappa_\lambda$ e Painel $2 \times 4$

In [ ]:
_ = plot_roc_per_kappa_individual(
    scores=eval_res["scores"],
    kappas=kgrid,
    save_dir=cmp_dir / "roc_per_kappa",
    show=True,
)

_ = plot_roc_and_pr_grid(
    scores=eval_res["scores"],
    kappas=kgrid,
    split_label=EVAL_SPLIT,
    save_path=cmp_dir / "roc_and_pr_grid.png",
    show=True,
)

_ = plot_metrics_bars_cmp(
    tables=eval_res["tables"],
    kappas=kgrid,
    split_label=EVAL_SPLIT,
    save_path=cmp_dir / "metrics_bars_comparison.png",
    show=True,
)

_ = plot_signal_eff_vs_bkg_rejection(
    scores=eval_res["scores"],
    kappas=kgrid,
    save_path=cmp_dir / "signal_eff_vs_bkg_rejection.png",
    show=True,
)

## 3.2 `AffinePNN` vs `DNN_AN` — Scores da Rede vs Eventos em Escala Log para cada $\kappa_\lambda$

In [ ]:
_ = plot_scores_vs_events_log_per_kappa(
    scores=eval_res["scores"],
    class_labels=cls_labels,
    kappas=kgrid,
    save_dir=cmp_dir / "scores_vs_events_log",
    show=True,
)

for k in kgrid:
    tag = str(k).replace(".", "p")
    _ = plot_score_events_log(
        scores=eval_res["scores"],
        kappa=float(k),
        class_labels=cls_labels,
        models=MODEL_NAMES,
        weighted=True,
        save_path=cmp_dir / "scores_vs_events_log" / f"stacked_yields_cmp_kl_{tag}.png",
        show=True,
    )

## 3.3 `AffinePNN` vs `DNN_AN` — Curvas Precision-Recall (Iso-F1), Barras PR/F1 e Pontos de Operação

In [ ]:
_ = plot_pr_curves_and_f1_summary(
    scores=eval_res["scores"],
    kappas=kgrid,
    save_dir=cmp_dir,
    show=True,
)

_ = plot_working_point_summary(
    wp_df,
    save_path=cmp_dir / "working_points_summary_cmp.png",
    show=True,
)

## 3.4 `AffinePNN` vs `DNN_AN` — Precision, Recall e F1-score vs Score da Rede (Separados e Comparando Todos)

In [ ]:
# 1) Plots SEPARADOS comparando AffinePNN vs DNN_AN (Precision, Recall e F1-score individuais)
_ = plot_prf1_vs_score_separate(
    scores=eval_res["scores"],
    kappas=kgrid,
    save_dir=cmp_dir / "prf1_vs_score_separate",
    show=True,
)

# 2) Plot ÚNICO comparando TODOS juntos (Precision + Recall + F1 vs Score: AffinePNN sólido vs DNN_AN tracejado)
_ = plot_prf1_vs_score_all_in_one(
    scores=eval_res["scores"],
    kappas=kgrid,
    save_path=cmp_dir / "prf1_vs_score_all_in_one_cmp.png",
    show=True,
)

## 3.5 `AffinePNN` vs `DNN_AN` — Matrizes de Confusão Comparativas por $\kappa_\lambda$

In [ ]:
for k in kgrid:
    tag = str(k).replace(".", "p")
    _ = plot_confusion_matrices(
        scores=eval_res["scores"],
        kappa=float(k),
        class_labels=cls_labels,
        models=MODEL_NAMES,
        save_path=cmp_dir / f"confusion_matrix_cmp_kl_{tag}.png",
        show=True,
    )

## 4. Significance Ratio $\sigma_{\text{ratio}}$ (Anzalone Eq. 4) e Rendimentos Físicos com Pesos NLO Assinados ($S/B$, $S/\sqrt{B}$, Asimov $Z_A$)

> **Nota Metodológica:** Enquanto as curvas ROC e Precision-Recall acima avaliam a forma das distribuições usando pesos não-negativos $w^+ = \max(w_{\text{phys}}, 0)$ (e reportam também a `AUC` sem peso), a tabela e o gráfico de rendimentos físicos abaixo preservam integralmente os **pesos NLO com sinal** ($N_{\text{esperado}} = \sum_i w_{\text{phys}, i}$).

In [ ]:
_ = plot_significance_ratio(
    scores=eval_res["scores"],
    kappas=kgrid,
    save_path=cmp_dir / "significance_ratio_curves.png",
    show=True,
)

_ = plot_physical_yields_and_significance(
    yields_tables=eval_res["yields"],
    sig_ratio_tables=eval_res["significance_ratio"],
    eff_target=0.80,
    save_path=cmp_dir / "physical_yields_and_significance_cmp.png",
    show=True,
)

for name in MODEL_NAMES:
    print(f"\n--- {name}: Significance Ratio (Anzalone Eq. 4) ---")
    display(eval_res["significance_ratio"][name].round(4))
    print(f"--- {name}: Rendimentos Físicos Esperados com Pesos NLO Assinados (DY + ttbar) ---")
    display(eval_res["yields"][name])

if eval_res.get("yields_comparison") is not None:
    print("\n--- Comparação Direta de Rendimentos Físicos e Significância de Asimov (AffinePNN vs DNN_AN) ---")
    display(eval_res["yields_comparison"].round(4))

## 5. Matriz de Resposta ($\kappa_{\text{verdadeiro}} \times \kappa_{\text{entrada}}$) e Varredura Contínua em $\kappa_\lambda \in [-2, 8]$

In [ ]:
resp_matrices = {}
for name in MODEL_NAMES:
    mat = kappa_response_matrix(
        model=models[name],
        ev_df=data.splits[EVAL_SPLIT],
        ev_xc=data.x_cont[EVAL_SPLIT],
        ev_xk=data.x_cat[EVAL_SPLIT],
        kappas=kgrid,
        kappa_norm=float(cfg["physics"]["kappa_norm"]),
        batch_size=int(cfg["evaluation"]["eval_batch_size"]),
    )
    resp_matrices[name] = mat
    mat.to_csv(out_paths.metrics / f"kappa_response_matrix_{name}_{EVAL_SPLIT}.csv")
    print(f"\nMatriz de Resposta AUC_w ({name}):")
    display(mat.round(4))

_ = plot_kappa_response_heatmap(
    resp_matrices,
    save_path=cmp_dir / f"kappa_response_heatmap_{EVAL_SPLIT}.png",
    show=True,
)

if "AffinePNN" in models:
    k_min, k_max, k_n = cfg["evaluation"]["kappa_scan_range"]
    scan_grid = np.linspace(float(k_min), float(k_max), int(k_n))
    scan_df = continuous_kappa_scan(
        model=models["AffinePNN"],
        ev_df=data.splits[EVAL_SPLIT],
        ev_xc=data.x_cont[EVAL_SPLIT],
        ev_xk=data.x_cat[EVAL_SPLIT],
        kappa_scan_values=scan_grid,
        ref_signal_kappas=kgrid,
        kappa_norm=float(cfg["physics"]["kappa_norm"]),
        batch_size=int(cfg["evaluation"]["eval_batch_size"]),
        seed=cfg["seed"],
    )
    scan_df.to_csv(out_paths.metrics / f"continuous_kappa_scan_{EVAL_SPLIT}.csv", index=False)

    _ = plot_continuous_kappa_scan(
        scan_df=scan_df,
        ref_kappas=kgrid,
        model_name="AffinePNN",
        save_path=pure_dir / "continuous_kappa_scan.png",
        show=True,
    )

---
## 6. Exportação Completa e Portável (`ONNX`, `ONNX End-to-End`, `SavedModel`, `.keras`, `.weights.h5`, `.npz`)

Esta seção empacota cada modelo em `outputs/exported_models/<Modelo>/` com todos os seus parâmetros e metadados:
- **`<Modelo>_end2end_raw.onnx`:** grafo ONNX autossuficiente para **CMSSW / Columnflow / C++ / Python** que recebe diretamente os valores físicos brutos das 49 features contínuas, 7 categóricas e $\kappa_\lambda$ físico (aplicando limpeza de sentinelas, mascaramento de `bjet2`/`fatjet`, `sign(x)*log1p(|x|)` na covariância da MET, padronização `z-score`, *lookup* categórico e divisão por `kappa_norm` dentro do próprio grafo ONNX!).
- **`<Modelo>.onnx`:** grafo ONNX sobre tensores já pré-processados.
- **`saved_model/` + `model_<Modelo>.keras` + `<Modelo>.weights.h5`:** formatos TensorFlow/Keras nativos.
- **`learned_parameters_<Modelo>.npz` + `architecture_<Modelo>.json` + `preprocessing_spec.json` + `bundle_manifest.json`:** todos os pesos internos (incluindo matrizes do LBN e escalas/biases afins), especificações e hashes SHA-256 com verificação automática via `onnxruntime`.

In [ ]:
manifests = {}
for name in MODEL_NAMES:
    manifests[name] = export_complete_model_bundle(
        model_name=name,
        model=models[name],
        scaler=data.scaler,
        cat_preprocessor=data.cat_preprocessor,
        cfg=cfg,
        export_root_dir=out_paths.exported_models,
        history_dict=histories.get(name),
        meta_dict=metas.get(name),
        opset=int(cfg.get("export", {}).get("onnx_opset", 17)),
        verify_parity=True,
    )
    p_ver = manifests[name]["onnx_parity_verification"]
    print(f"=== Pacote Exportado: {name} ===")
    print("Diretório :", manifests[name]["bundle_dir"])
    print("Artefatos :", json.dumps(manifests[name]["artifacts"], indent=2))
    print(
        f"Paridade Keras vs ONNX Runtime ({p_ver['n_test_samples']} eventos): "
        f"max|ΔP| pre={p_ver['preprocessed_onnx_max_abs_diff']:.2e} | "
        f"end2end_raw={p_ver['end2end_raw_onnx_max_abs_diff']:.2e} [{p_ver['status']}]\n"
    )

# Demonstração rápida: inferência pura via ONNX Runtime sem depender de Keras/TensorFlow
onnx_pred = ONNXModelPredictor(out_paths.exported_models / "AffinePNN", use_end2end_raw=True)
sample_probs = onnx_pred.predict_raw_dataframe(data.splits[EVAL_SPLIT].iloc[:5], kappa_lambda=1.0)
print("Exemplo de predição [P(DY), P(ttbar), P(HH)] com ONNX End-to-End (κ_λ = 1.0):")
display(pd.DataFrame(sample_probs, columns=["P(DY)", "P(ttbar)", "P(HH)"]))

## 7. Checklist dos 9 Estudos Físicos e Estatísticos para a Dissertação

- [ ] **1. ROC e PR por hipótese de $\kappa_\lambda$:** `AUC_w` (pesos não-negativos $w^+ = \max(w_{\text{phys}}, 0)$), `AUC` sem peso e `PR-AUC` avaliados em $\kappa_\lambda \in \{0, 1, 2.45, 5\}$ (`Seções 1, 2.1, 2.3, 3.1, 3.3`).
- [ ] **2. Eficiência de sinal e rejeição de background:** curvas $\epsilon_S$ vs $1/\epsilon_B$ em log e razão de rejeição $(1/\epsilon_B)_{\text{Affine}} / (1/\epsilon_B)_{\text{AN}}$ (`Seção 3.1` + `wp_df`).
- [ ] **3. Distribuições de $P(\text{HH})$:** scores em escala log por classe (`DY`, `ttbar`, `HH`) em eventos MC brutos e rendimentos ponderados (`Seções 2.2 e 3.2`).
- [ ] **4. Estudo de interpolação em $\kappa_\lambda = 2.45$:** treino omitindo `held_out = 2.45` comparando `AffinePNN` interpolada vs `PNN_Concat`, `DNN_AN` e $\kappa_{\text{in}}$ incorretos (`Notebook 04 Seção 6` / `sanity_studies.py --study interpolation`).
- [ ] **5. Matriz de resposta entre $\kappa_{\text{verdadeiro}}$ e $\kappa_{\text{entrada}}$:** diagonal ótima na `AffinePNN` e colunas idênticas no controle `DNN_AN` (`Seção 5` e `Notebook 04 Seção 4`).
- [ ] **6. Estudo de overtraining:** teste Kolmogorov-Smirnov + histogramas sobrepostos Treino vs Val/Test com razão por bin e Fig. 50 da AN (`Notebook 04 Seção 2` / `scripts/evaluate.py`).
- [ ] **7. Teste de random labels:** loss de treino estabilizada em $\ln(3) \approx 1.0986$ e $\text{AUC} \approx 0.50$ (`Notebook 04 Seção 6` / `sanity_studies.py --study random_label`).
- [ ] **8. Comparação da estabilidade entre seeds:** média $\pm 1\sigma$ de $\text{AUC}_w$ e do ganho $\Delta\text{AUC}_w$ entre múltiplas seeds (`Notebook 04 Seção 6` / `sanity_studies.py --study seeds`).
- [ ] **9. Comparação dos rendimentos físicos e significâncias:** $S$, $B$, $S/B$, $S/\sqrt{B}$ e significância de Asimov $Z_A$ calculados com os **pesos NLO assinados originais** ($\sum_i w_{\text{phys}, i}$), além de $\sigma_{\text{ratio}}$ (`Seção 4`).